In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'data').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / 'data').exists():
    raise FileNotFoundError('Não foi possível localizar a raiz do projeto.')

for candidate in [
    PROJECT_ROOT / 'code',
    PROJECT_ROOT / 'code' / 'revenue',
    PROJECT_ROOT / 'code' / 'tmdb',
]:
    if candidate.exists() and str(candidate.resolve()) not in sys.path:
        sys.path.append(str(candidate.resolve()))

import pandas as pd

from imbalance_experiment_utils import (
    HYBRID_CLASSIFICATION_REGRESSION_ARTIFACT_DIR,
    HYBRID_CLASSIFIER_CONFIGS,
    REVENUE_BAND_LABELS,
    TMDB_EXTENDED_NO_TRANSFORM_ARTIFACT_DIR,
    load_best_params_lookup_from_artifact_dir,
    load_results_from_artifact_dir,
    load_tmdb_extended_context,
    run_hybrid_classification_regression_experiment,
    save_additional_table,
    save_artifact_tables,
    summarize_errors_by_band,
)


/home/gabriel/Faculdade/Matérias/ML/UFSJ_Aprendizado_Maquina_TP1/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# **Abordagem Híbrida na Base TMDB Estendida: Classificação + Regressão**

Este notebook implementa uma estratégia em dois estágios: primeiro um classificador prevê a faixa de arrecadação do filme; em seguida, um regressor local especializado na faixa prevista estima o valor contínuo de `revenue`. Diferentemente do notebook `06`, aqui não há uso da faixa verdadeira no teste.

In [2]:
TARGET_NAME = 'Sem transformação'
OVERWRITE_ARTIFACTS = False
SHOW_PROGRESS = True
MIN_BAND_SAMPLES = 80

ARTIFACT_DIR = HYBRID_CLASSIFICATION_REGRESSION_ARTIFACT_DIR
ERROR_ANALYSIS_DIR = ARTIFACT_DIR / 'error_analysis'
BEST_COMPARISON_PATH = ARTIFACT_DIR / 'best_hybrid_vs_global_best.csv'
CONFUSION_PATH = ERROR_ANALYSIS_DIR / '08_classification_plus_regression_confusion_matrix.csv'
BAND_METRICS_PATH = ERROR_ANALYSIS_DIR / '08_classification_plus_regression_metricas_por_faixa.csv'
TRAINING_SIZES_PATH = ARTIFACT_DIR / 'training_band_sizes.csv'


In [3]:
context = load_tmdb_extended_context()
df_movies = context['df_movies']
X = context['X']
y = context['y']
folds_df = context['folds_df']
revenue_bins = context['revenue_bins']

print(f'Base TMDB estendida: {df_movies.shape[0]} filmes | {X.shape[1]} features')
display(df_movies[['id_tmdb', 'title', 'revenue']].head())


Base TMDB estendida: 6918 filmes | 284 features


,id_tmdb,title,revenue
0,552524,Lilo & Stitch,610800000
1,950387,A Minecraft Movie,947000000
2,1257960,सिकंदर,24727058
3,574475,Final Destination Bloodlines,229314062
4,1197306,A Working Man,98652557


In [4]:
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
ERROR_ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

if (
    not OVERWRITE_ARTIFACTS
    and (ARTIFACT_DIR / 'model_selection_results.csv').exists()
    and (ARTIFACT_DIR / 'model_selection_predictions.csv').exists()
    and (ARTIFACT_DIR / 'model_selection_summary.csv').exists()
    and TRAINING_SIZES_PATH.exists()
):
    results_df, predictions_df, summary_df = load_results_from_artifact_dir(ARTIFACT_DIR)
    training_sizes_df = pd.read_csv(TRAINING_SIZES_PATH)
else:
    best_params_lookup = load_best_params_lookup_from_artifact_dir(
        TMDB_EXTENDED_NO_TRANSFORM_ARTIFACT_DIR,
        target_name=TARGET_NAME,
    )
    results_df, predictions_df, summary_df, training_sizes_df = run_hybrid_classification_regression_experiment(
        df_movies=df_movies,
        X=X,
        y=y,
        folds_df=folds_df,
        revenue_bins=revenue_bins,
        best_params_lookup=best_params_lookup,
        classifier_configs=HYBRID_CLASSIFIER_CONFIGS,
        target_name=TARGET_NAME,
        min_band_samples=MIN_BAND_SAMPLES,
        show_progress=SHOW_PROGRESS,
    )
    save_artifact_tables(
        ARTIFACT_DIR,
        results_df=results_df,
        predictions_df=predictions_df,
        summary_df=summary_df,
    )
    training_sizes_df.to_csv(TRAINING_SIZES_PATH, index=False)

global_results_df, global_predictions_df, global_summary_df = load_results_from_artifact_dir(
    TMDB_EXTENDED_NO_TRANSFORM_ARTIFACT_DIR,
)
global_summary_df = global_summary_df.loc[global_summary_df['target_version'] == TARGET_NAME].copy()
summary_df = summary_df.loc[summary_df['target_version'] == TARGET_NAME].copy()

best_hybrid_row = summary_df.sort_values(['mean_rmse', 'mean_mae', 'model']).iloc[0]
best_global_row = global_summary_df.sort_values(['mean_rmse', 'mean_mae', 'model']).iloc[0]

best_comparison_df = pd.DataFrame([
    {
        'cenário': 'Melhor híbrido',
        'modelo': best_hybrid_row['model'],
        'mean_rmse': best_hybrid_row['mean_rmse'],
        'mean_mae': best_hybrid_row['mean_mae'],
        'mean_r2': best_hybrid_row['mean_r2'],
        'mean_band_macro_f1': best_hybrid_row['mean_band_macro_f1'],
    },
    {
        'cenário': 'Melhor global TMDB estendido',
        'modelo': best_global_row['model'],
        'mean_rmse': best_global_row['mean_rmse'],
        'mean_mae': best_global_row['mean_mae'],
        'mean_r2': best_global_row['mean_r2'],
        'mean_band_macro_f1': None,
    },
])
best_comparison_df['delta_rmse_vs_global'] = best_comparison_df['mean_rmse'] - best_global_row['mean_rmse']
best_comparison_df['delta_mae_vs_global'] = best_comparison_df['mean_mae'] - best_global_row['mean_mae']
best_comparison_df['delta_r2_vs_global'] = best_comparison_df['mean_r2'] - best_global_row['mean_r2']
save_additional_table(best_comparison_df, BEST_COMPARISON_PATH)

best_hybrid_predictions_df = predictions_df.loc[predictions_df['model'] == best_hybrid_row['model']].copy()
band_metrics_df = summarize_errors_by_band(best_hybrid_predictions_df, revenue_bins)
save_additional_table(band_metrics_df, BAND_METRICS_PATH)

confusion_df = pd.crosstab(
    pd.Categorical(best_hybrid_predictions_df['true_band'], categories=REVENUE_BAND_LABELS, ordered=True),
    pd.Categorical(best_hybrid_predictions_df['predicted_band'], categories=REVENUE_BAND_LABELS, ordered=True),
    normalize='index',
)
confusion_df = confusion_df.reindex(index=REVENUE_BAND_LABELS, columns=REVENUE_BAND_LABELS, fill_value=0.0)
confusion_df.index.name = 'true_band'
save_additional_table(confusion_df.reset_index(), CONFUSION_PATH)

display(summary_df[[
    'model',
    'classifier_model',
    'regressor_model',
    'mean_rmse',
    'mean_mae',
    'mean_r2',
    'mean_band_macro_f1',
]])
display(best_comparison_df)
display(band_metrics_df)
display(confusion_df)


Classificação + regressão: 100%|██████████| 590/590 [32:49<00:00,  3.34s/ajuste, XGBoost Classifier | fold 9 | melhor=0.4863 | refit=sim]            


,model,classifier_model,regressor_model,mean_rmse,mean_mae,mean_r2,mean_band_macro_f1
0,Gradient Boosting Classifier + Gradient Boosti...,Gradient Boosting Classifier,Gradient Boosting Regressor,1.178444e+08,5.889427e+07,0.586326,0.469628
1,XGBoost Classifier + XGBoost Regressor,XGBoost Classifier,XGBoost Regressor,1.180109e+08,5.861306e+07,0.584359,0.468302
2,Random Forest Classifier + Random Forest Regre...,Random Forest Classifier,Random Forest Regressor,1.217155e+08,6.122261e+07,0.553342,0.444191


,cenário,modelo,mean_rmse,mean_mae,mean_r2,mean_band_macro_f1,delta_rmse_vs_global,delta_mae_vs_global,delta_r2_vs_global
0,Melhor híbrido,Gradient Boosting Classifier + Gradient Boosti...,1.178444e+08,5.889427e+07,0.586326,0.469628,6.179461e+06,3.736758e+06,-0.043849
1,Melhor global TMDB estendido,XGBoost Regressor,1.116649e+08,5.515751e+07,0.630175,NaN,0.000000e+00,0.000000e+00,0.000000


,faixa_receita,quantidade_filmes,receita_minima,receita_maxima,mae_medio,residuo_medio,mediana_erro_absoluto,percentual_subestimados,percentual_superestimados,rmse
0,Muito baixa receita,1384,1,7086000,1.353184e+07,-1.248996e+07,2.456152e+06,27.312139,72.687861,4.090203e+07
1,Baixa receita,1383,7096000,22441323,2.480127e+07,-1.754000e+07,1.098091e+07,44.107014,55.892986,5.095206e+07
2,Média receita,1384,22468044,52800000,3.930517e+07,-1.874411e+07,2.712171e+07,50.433526,49.566474,6.656251e+07
3,Alta receita,1383,52900000,134038006,5.995716e+07,-1.178081e+07,3.713796e+07,57.483731,42.516269,8.887672e+07
4,Muito alta receita,1384,134100000,2923706026,1.568543e+08,4.647431e+07,1.170778e+08,60.404624,39.595376,2.316398e+08


col_0,Muito baixa receita,Baixa receita,Média receita,Alta receita,Muito alta receita
true_band,,,,,
Muito baixa receita,0.650289,0.192197,0.097543,0.050578,0.009393
Baixa receita,0.290672,0.328272,0.208966,0.150398,0.021692
Média receita,0.135116,0.242775,0.258671,0.305636,0.057803
Alta receita,0.050615,0.107737,0.198843,0.451193,0.191612
Muito alta receita,0.012283,0.036127,0.065751,0.203035,0.682803
